# Librerias

In [1]:
import time
import math
import requests
from bs4 import BeautifulSoup
import pandas as pd
from collections import namedtuple
import re
#from sentence_transformers import SentenceTransformer, util
import time
from tqdm import tqdm
from collections import namedtuple
import random
import numpy as np

from datetime import date
import json
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [2]:
 date.today()

datetime.date(2026, 4, 3)

# Lista de urls

In [3]:
base = "https://inmuebles.mercadolibre.com.mx"

tipos = ["casas", "departamentos"]
operaciones = ["venta", "renta"]

# -------------------------
# 1) CDMX: 16 alcaldías
# -------------------------
cdmx_alcaldias = [
    "alvaro-obregon",
    "azcapotzalco",
    "benito-juarez",
    "coyoacan",
    "cuajimalpa-de-morelos",
    "cuauhtemoc",
    "gustavo-a-madero",
    "iztacalco",
    "iztapalapa",
    "la-magdalena-contreras",
    "miguel-hidalgo",
    "milpa-alta",
    "tlahuac",
    "tlalpan",
    "venustiano-carranza",
    "xochimilco",
]

# -------------------------
# 2) EdoMex metropolitano
# -------------------------
edomex_metro = [
    "atizapan-de-zaragoza",
    "coacalco-de-berriozabal",
    "cuautitlan",
    "cuautitlan-izcalli",
    "chalco",
    "chicoloapan",
    "chimalhuacan",
    "ecatepec-de-morelos",
    "huixquilucan",
    "ixtapaluca",
    "la-paz",
    "naucalpan-de-juarez",
    "nezahualcoyotl",
    "nicolas-romero",
    "tecamac",
    "teoloyucan",
    "tepotzotlan",
    "texcoco",
    "tlalnepantla-de-baz",
    "tultitlan",
    "valle-de-chalco-solidaridad",
    "zumpango",
]

# -------------------------
# 3) Estados a nivel general
# -------------------------
estados_generales = [
    "aguascalientes",
    "baja-california",
    "baja-california-sur",
    "campeche",
    "chiapas",
    "chihuahua",
    "coahuila",
    "colima",
    "durango",
    "guanajuato",
    "guerrero",
    "hidalgo",
    "jalisco",
    "michoacan",
    "morelos",
    "nayarit",
    "nuevo-leon",
    "oaxaca",
    "puebla",
    "queretaro",
    "quintana-roo",
    "san-luis-potosi",
    "sinaloa",
    "sonora",
    "tabasco",
    "tamaulipas",
    "tlaxcala",
    "veracruz",
    "yucatan",
    "zacatecas",
]

# -------------------------
# 4) Ciudades / municipios grandes
#    (para scraping más fino)
# -------------------------
ciudades_grandes = {
    "aguascalientes": [
        "aguascalientes",
        "jesus-maria",
    ],
    "baja-california": [
        "tijuana",
        "mexicali",
        "ensenada",
        "tecate",
        "playas-de-rosarito",
    ],
    "baja-california-sur": [
        "la-paz",
        "los-cabos",
    ],
    "campeche": [
        "campeche",
        "carmen",
    ],
    "chiapas": [
        "tuxtla-gutierrez",
        "san-cristobal-de-las-casas",
        "tapachula",
    ],
    "chihuahua": [
        "chihuahua",
        "juarez",
        "delicias",
    ],
    "coahuila": [
        "saltillo",
        "torreon",
        "monclova",
        "piedras-negras",
        "ramos-arizpe",
    ],
    "colima": [
        "colima",
        "manzanillo",
        "villa-de-alvarez",
    ],
    "durango": [
        "durango",
        "gomez-palacio",
        "lerdo",
    ],
    "guanajuato": [
        "leon",
        "celaya",
        "irapuato",
        "salamanca",
        "guanajuato",
        "san-miguel-de-allende",
    ],
    "guerrero": [
        "acapulco-de-juarez",
        "chilpancingo-de-los-bravo",
        "iguala-de-la-independencia",
    ],
    "hidalgo": [
        "pachuca-de-soto",
        "mineral-de-la-reforma",
        "tizayuca",
        "tula-de-allende",
    ],
    "jalisco": [
        "guadalajara",
        "zapopan",
        "tlajomulco-de-zuniga",
        "san-pedro-tlaquepaque",
        "tonala",
        "puerto-vallarta",
    ],
    "michoacan": [
        "morelia",
        "uruapan",
        "zamora",
        "lazaro-cardenas",
    ],
    "morelos": [
        "cuernavaca",
        "jiutepec",
        "temixco",
        "yautepec",
        "emiliano-zapata",
        "xochitepec",
        "cuautla",
    ],
    "nayarit": [
        "tepic",
        "bahia-de-banderas",
    ],
    "nuevo-leon": [
        "monterrey",
        "guadalupe",
        "san-nicolas-de-los-garza",
        "apodaca",
        "general-escobedo",
        "santa-catarina",
        "san-pedro-garza-garcia",
        "garcia",
        "juarez",
    ],
    "oaxaca": [
        "oaxaca-de-juarez",
        "santa-cruz-xoxocotlan",
        "san-juan-bautista-tuxtepec",
    ],
    "puebla": [
        "puebla",
        "san-andres-cholula",
        "san-pedro-cholula",
        "cuautlancingo",
        "tehuacan",
        "coronango",
        "atlixco",
    ],
    "queretaro": [
        "queretaro",
        "el-marques",
        "corregidora",
        "san-juan-del-rio",
    ],
    "quintana-roo": [
        "benito-juarez",
        "solidaridad",
        "tulum",
        "othon-p-blanco",
        "isla-mujeres",
        "puerto-morelos",
    ],
    "san-luis-potosi": [
        "san-luis-potosi",
        "soledad-de-graciano-sanchez",
    ],
    "sinaloa": [
        "culiacan",
        "mazatlan",
        "ahome",
        "guasave",
    ],
    "sonora": [
        "hermosillo",
        "cajeme",
        "nogales",
        "san-luis-rio-colorado",
    ],
    "tabasco": [
        "centro",
        "paraiso",
        "comalcalco",
    ],
    "tamaulipas": [
        "reynosa",
        "matamoros",
        "nuevo-laredo",
        "tampico",
        "ciudad-madero",
        "altamira",
        "victoria",
    ],
    "tlaxcala": [
        "tlaxcala",
        "apizaco",
        "chiautempan",
        "huamantla",
    ],
    "veracruz": [
        "veracruz",
        "boca-del-rio",
        "xalapa",
        "coatzacoalcos",
        "cordoba",
        "orizaba",
        "poza-rica-de-hidalgo",
    ],
    "yucatan": [
        "merida",
        "progreso",
        "valladolid",
    ],
    "zacatecas": [
        "zacatecas",
        "guadalupe",
        "fresnillo",
    ],
}

# -------------------------
# Generación de URLs
# -------------------------
urls = []

# CDMX
urls += [
    f"{base}/{tipo}/{operacion}/distrito-federal/{alcaldia}/"
    for alcaldia in cdmx_alcaldias
    for tipo in tipos
    for operacion in operaciones
]

# EdoMex metropolitano
urls += [
    f"{base}/{tipo}/{operacion}/estado-de-mexico/{municipio}/"
    for municipio in edomex_metro
    for tipo in tipos
    for operacion in operaciones
]

# Estados generales
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/"
    for estado in estados_generales
    for tipo in tipos
    for operacion in operaciones
]

# Ciudades / municipios grandes
urls += [
    f"{base}/{tipo}/{operacion}/{estado}/{ciudad}/"
    for estado, ciudades in ciudades_grandes.items()
    for ciudad in ciudades
    for tipo in tipos
    for operacion in operaciones
]

# quitar duplicados por si algo se repite
urls = list(dict.fromkeys(urls))

print(f"Total URLs: {len(urls)}")
print(urls[:20])

Total URLs: 776
['https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/alvaro-obregon/', 'https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/alvaro-obregon/', 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/alvaro-obregon/', 'https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/alvaro-obregon/', 'https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/azcapotzalco/', 'https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/azcapotzalco/', 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/azcapotzalco/', 'https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/azcapotzalco/', 'https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/benito-juarez/', 'https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/benito-juarez/', 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/benito-juarez/', 'https:/

In [ ]:
urls[0]

# Funciones

In [4]:
import os

ruta_archivo = "ws_vivienda.csv"

def guardar_dataframe(df, ruta):
    
    # si el archivo ya existe no escribimos header
    if os.path.exists(ruta):
        df.to_csv(ruta, mode='a', header=False, index=False)
    else:
        df.to_csv(ruta, mode='w', header=True, index=False)

In [5]:
import re

def extraer_datos_url(url):
    # caso con estado y municipio
    m1 = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/([^/]+)/', url)
    if m1:
        tipo_vivienda, operacion, estado, municipio = m1.groups()
        return tipo_vivienda, operacion, estado, municipio

    # caso con solo estado
    m2 = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/', url)
    if m2:
        tipo_vivienda, operacion, estado = m2.groups()
        return tipo_vivienda, operacion, estado, None

    return None, None, None, None

## Funciones Principales

In [6]:
def limpia_precio(txt):
    if not txt: 
        return None
    # quita símbolos comunes de MercadoLibre (puntos de miles, comas, $)
    t = txt.replace("\xa0", " ").replace(".", "").replace(",", "").replace("$", "").strip()
    # a veces viene junto con centavos en otro span; nos quedamos con entero
    try:
        return int("".join(ch for ch in t if ch.isdigit()))
    except ValueError:
        return None
import re

def extrae_atributos(li):
    """
    Regresa dict con recamaras, banos, sup_m2, sup_construida_m2 (si aparece), etc.
    Toma los <li> dentro de ul.poly-attributes_list
    """
    attrs = {
        "recamaras": None,
        "banos": None,
        "superficie_m2": None,
        "superficie_construida_m2": None
    }

    items = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    textos = [it.get_text(" ", strip=True).lower() for it in items]

    for t in textos:
        # recámaras / habitaciones / dormitorios
        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", t)
        if m:
            attrs["recamaras"] = int(m.group(1))
            continue

        # baños
        m = re.search(r"(\d+)\s*(baños|banos)", t)
        if m:
            attrs["banos"] = int(m.group(1))
            continue

        # superficie construida
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²\s*(construidos|construidas|construido|construida)", t)
        if m:
            attrs["superficie_construida_m2"] = float(m.group(1))
            continue

        # superficie (genérica) en m²
        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", t)
        if m and attrs["superficie_m2"] is None:
            attrs["superficie_m2"] = float(m.group(1))
            continue

    return attrs

def parse_item(li):
    # Títulos
    fecha =  date.today()
    
        
    descripcion = None
    link = None
    a = li.select_one("h3.poly-component__title-wrapper a.poly-component__title[href]")
    if a:
        descripcion = a.get_text(strip=True)
        link = a["href"]
        

    # Imagen
    img = li.find("img")
    imagen = img.get("data-src") or img.get("src") if img else None

    # Precios (ML usa varias clases; probamos alternativas)
    precio_actual = None
    precio_antes = None

    # Número entero del precio actual
    actual_span = li.select_one(".poly-price__current .andes-money-amount__fraction")
    if actual_span:
        precio_actual = limpia_precio(actual_span.get_text())

    # Posible precio anterior (tachado) aparece con otra clase o data-test-id
    antes = (li.select_one(".andes-money-amount.comparative-price .andes-money-amount__fraction")
             or li.select_one("[data-test-id='price-before'] .andes-money-amount__fraction")
             or li.select_one(".ui-search-price__second-line .price-tag__disabled .price-tag-amount"))
    if antes:
        precio_antes = limpia_precio(antes.get_text())
    # -------- ATRIBUTOS NUEVOS (recámaras, baños, superficie) --------
    recamaras = None
    banos = None
    superficie_m2 = None

    attrs = li.select("ul.poly-attributes_list li.poly-attributes_list__item")
    for it in attrs:
        txt = it.get_text(" ", strip=True).lower()

        m = re.search(r"(\d+)\s*(recámaras|recamaras|habitaciones|dormitorios)", txt)
        if m:
            recamaras = int(m.group(1))
            continue

        m = re.search(r"(\d+)\s*(baños|banos)", txt)
        if m:
            banos = int(m.group(1))
            continue

        m = re.search(r"(\d+(?:\.\d+)?)\s*m²", txt)
        if m:
            superficie_m2 = float(m.group(1))
            continue

    return Articulo(
        date.today(),
        descripcion,
        recamaras,
        banos,
        superficie_m2,
        precio_actual,
        link,
        imagen
    )

    

Articulo = namedtuple("Articulo", ["fecha", "descripcion", "recamaras", "banos", "superficie_m2", "precio_actual", "link", "imagen"])
articulos = []

# ---------- Config ----------
PAIS = "com.mx"  # cambia a "com.co", "com.ar", etc.

PAGINAS = 1            # cuántas páginas raspar
ESPERA_S = 30         # pausa entre páginas (ser buen ciudadano)
BASE = f"https://listado.mercadolibre.{PAIS}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-MX,es;q=0.9"
}




## Función de coordenadas

In [7]:
import requests
import re
import pandas as pd

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-MX,es;q=0.9"
}


def get_lat_lon(url, headers=HEADERS):
    url = url.split("?")[0].split("#")[0]
    resp = requests.get(url, headers=headers, timeout=25)
    html = resp.text

    m = re.search(
        r'"map_info".*?"location":\{"latitude":"?([-0-9.]+)"?,"longitude":"?([-0-9.]+)"?\}',
        html,
        flags=re.DOTALL
    )

    if m:
        lat = float(m.group(1))
        lon = float(m.group(2))
        return lat, lon

    print("No se encontraron coordenadas")
    return None, None


# ---------------- USO ---------------- #

#url = "https://departamento.mercadolibre.com.mx/MLM-2433508491-departamento-en-venta-en-paseos-de-taxquena-coyoacan-_JM#polycard_client=search-nordic&search_layout=grid&position=1&type=item&tracking_id=53c09837-a3f1-45f4-bc99-b8ff7aba6b83"

#lat, lon = get_lat_lon(url)

# DataFrame con SOLO coordenadas
#df = pd.DataFrame([{
#    "latitud": lat,
#    "longitud": lon
#}])

#df

## Función de descripción

In [18]:
import requests
from bs4 import BeautifulSoup

def get_descripcion(url, headers=HEADERS):
    """
    Devuelve la descripción completa del anuncio de MercadoLibre.
    Retorna None si no se encuentra.
    """
    try:
        resp = requests.get(url, headers=headers, timeout=25)
        resp.raise_for_status()
    except Exception as e:
        print(f"[WARN] Error al obtener la descripción: {e}")
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    # Primer intento: selector principal
    desc_node = soup.select_one("p.ui-pdp-description__content")

    # Segundo intento (por si cambia la clase)
    if not desc_node:
        desc_node = soup.select_one("[data-testid='content']")

    if not desc_node:
        print("[INFO] No se encontró la descripción en la página.")
        return None

    # Extraemos texto limpio
    descripcion = desc_node.get_text(separator=" ", strip=True)
    time.sleep(random.randint(30, 60)) 
    return descripcion

## Función de Scrap

In [15]:
# ---------- Scrape con paginado ----------
def scrap_page(PAGINAS,URL,HEADERS):
    for page in range(1, PAGINAS + 1):
        print(f'Scrapeando Página {page}')
        # En ML el paginado suele ser con _Desde_ o _Desde_XX; esta variante funciona en varios países:
        sep = "" if URL.endswith("/") else "/"
        url_pag = URL if page == 1 else f"{URL}{sep}_Desde_{(page-1)*48+1}"
        try:
            resp = requests.get(url_pag, headers=HEADERS, timeout=20)
            resp.raise_for_status()
        except requests.HTTPError as e:
            print(f"[WARN] {e} -> {url_pag}")
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        # Cada item suele estar en <li> del layout de búsqueda
        items = soup.select("li.ui-search-layout__item, li.ui-search-layout__item--stack")
        if not items:
            # fallback genérico
            items = soup.select("li.ui-search-result__wrapper")
        if not items:
            print(f"[WARN] No se hallaron items en página {page}.")
            break

        for li in items:
            try:
                articulos.append(parse_item(li))
            except Exception as exc:
                # si un item rompe, sigue con el siguiente
                print(f"[WARN] item con error: {exc}")

        time.sleep(ESPERA_S)
        df = pd.DataFrame(articulos)
      

        #time.sleep(random.randint(60, 120))  # MUY importante para no bloquearte

        return df

# Scrapeo princial

In [ ]:
#QUERY = list(dfp.product_name)[i]  # tu búsqueda
#URL = f"{BASE}/{QUERY.replace(' ', '-')}"
URL ='https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/'
articulos = []
print(URL)
df_scrap=scrap_page(1,URL,HEADERS)#[["nombre", "descripcion", "precio_actual"]]
print (f'se escrapearon {df_scrap.shape[0]} articulos')

In [ ]:
df_scrap

In [ ]:
df_scrap

# Dataframe con Coordenadas

In [ ]:
url= df_scrap.link[9]
lat, lon = get_lat_lon(url)
print(lat)
print(lon)

In [ ]:
df_crd  =pd.DataFrame(columns=["latitud","longitud"])
df_crd

In [ ]:
url= df_scrap.link[0]
print(url)
print(get_lat_lon(url))

In [ ]:
url= df_scrap.link[1]
print(url)
print(get_lat_lon(url))

In [ ]:
url= df_scrap.link[3]
print(url)
print(get_lat_lon(url))

In [ ]:
# DataFrame con SOLO coordenadas
df_crd  =pd.DataFrame(columns=["latitud","longitud"])
for i in range(df_scrap.shape[0]):  #df_scrap.shape[0]
    url= df_scrap.link[i]
    lat, lon = get_lat_lon(url)

    # DataFrame con SOLO coordenadas
    df = pd.DataFrame([{
        "latitud": lat,
        "longitud": lon
    }])

    df_crd= pd.concat([df_crd, df], ignore_index=True)
    time.sleep(random.randint(15, 30))

In [ ]:
df_crd

In [ ]:
df_scrap[['atitud', 'longitud']] = df_crd[['latitud', 'longitud']]

In [ ]:
df_scrap

# Dataframe Descripción

In [ ]:
df_desc 

In [ ]:
# DataFrame con SOLO coordenadas
df_desc =pd.DataFrame(columns=['descripcion'])
for i in range(df_scrap.shape[0]):
    url= df_scrap.link[i]
    desc = get_descripcion(url)

    # DataFrame con SOLO coordenadas
    df = pd.DataFrame([{
        "descripcion": desc
    }])

    df_desc= pd.concat([df_desc, df], ignore_index=True)
    time.sleep(random.randint(15, 30))

In [ ]:
df_desc.descripcion[2]

In [ ]:
df_desc

In [ ]:
df_scrap[['descripcion']] = df_desc[['descripcion']]

In [ ]:
df_scrap

In [ ]:
latlon = get_lat_lon(df_scrap.link[0])
print(latlon)

In [ ]:
latlon = get_lat_lon(df_scrap.link[1])
print(latlon)

# Web Scraping en conjunto

In [ ]:
#QUERY = list(dfp.product_name)[i]  # tu búsqueda
#URL = f"{BASE}/{QUERY.replace(' ', '-')}"
paginas = 1
URL ='https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/coyoacan/'
articulos = []
print(URL)
df_scrap=scrap_page(paginas,URL,HEADERS)#[["nombre", "descripcion", "precio_actual"]]
print (f'se escrapearon {df_scrap.shape[0]} articulos')

# DataFrame con SOLO coordenadas
df_crd  =pd.DataFrame(columns=["latitud","longitud"])
for i in range(df_scrap.shape[0]):  #df_scrap.shape[0]
    url= df_scrap.link[i]
    lat, lon = get_lat_lon(url)

    # DataFrame con SOLO coordenadas
    df = pd.DataFrame([{
        "latitud": lat,
        "longitud": lon
    }])

    df_crd= pd.concat([df_crd, df], ignore_index=True)
    time.sleep(random.randint(15, 30))
df_scrap[['atitud', 'longitud']] = df_crd[['latitud', 'longitud']]
# DataFrame con SOLO coordenadas
df_desc =pd.DataFrame(columns=['descripcion'])
for i in range(df_scrap.shape[0]):
    url= df_scrap.link[i]
    desc = get_descripcion(url)

    # DataFrame con SOLO coordenadas
    df = pd.DataFrame([{
        "descripcion": desc
    }])

    df_desc= pd.concat([df_desc, df], ignore_index=True)
    time.sleep(random.randint(15, 30))
df_scrap[['descripcion']] = df_desc[['descripcion']]


In [ ]:
df_scrap.columns

# Corriendo URLs

In [ ]:
import random
urls1 = random.sample(urls, 5)
print(urls1)

In [12]:
urls1=urls[21:]

In [11]:
urls[21:]

['https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/cuauhtemoc/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/cuauhtemoc/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/cuauhtemoc/',
 'https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/gustavo-a-madero/',
 'https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/gustavo-a-madero/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/gustavo-a-madero/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/gustavo-a-madero/',
 'https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/iztacalco/',
 'https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/iztacalco/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/iztacalco/',
 'https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/iztacalco/',
 'https://inmuebles

In [ ]:
urls12 = ['https://inmuebles.mercadolibre.com.mx/departamentos/venta/sonora/']

In [ ]:
i = 0
paginas = 4

for url_base in tqdm(urls1):
    URL = url_base
    print(URL)
    print(f'Este es el url en la posicion {i}')

    try:
        df_scrap = scrap_page(paginas, URL, HEADERS)
    except Exception as e:
        print(f"[ERROR] Falló scrap_page en {URL}: {e}")
        continue

    # Validar si no regresó nada
    if df_scrap is None or df_scrap.empty:
        print(f"[WARN] No se obtuvo información para {URL}")
        continue

    print(f"se escrapearon {df_scrap.shape[0]} articulos")

    # Coordenadas
    df_crd = pd.DataFrame(columns=["latitud", "longitud"])

    for j in range(df_scrap.shape[0]):
        print(f"coorendas de la vivienda {j}, de {df_scrap.shape[0]} viviendas")
        link_articulo = df_scrap.loc[j, 'link']
        lat, lon = get_lat_lon(link_articulo)

        df = pd.DataFrame([{
            "latitud": lat,
            "longitud": lon
        }])

        df_crd = pd.concat([df_crd, df], ignore_index=True)
        time.sleep(random.randint(15, 30))

    df_scrap[['latitud', 'longitud']] = df_crd[['latitud', 'longitud']]

    # Descripciones
    df_desc = pd.DataFrame(columns=["descripcion"])

    for j in range(df_scrap.shape[0]):
        link_articulo = df_scrap.loc[j, 'link']
        print(f"descripcion de la vivienda {j}, de {df_scrap.shape[0]} viviendas")
        desc = get_descripcion(link_articulo)

        df = pd.DataFrame([{
            "descripcion": desc
        }])

        df_desc = pd.concat([df_desc, df], ignore_index=True)
        time.sleep(random.randint(15, 30))

    df_scrap[['descripcion']] = df_desc[['descripcion']]
    tipo_vivienda, operacion, estado, municipio = extraer_datos_url(URL)
    #tipo_vivienda, operacion, estado, municipio = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/([^/]+)/',URL).groups()
    df_scrap['tipo_vivienda'] = tipo_vivienda
    df_scrap['operacion'] = operacion
    df_scrap['estado'] = estado
    df_scrap['municipio'] = municipio
    guardar_dataframe(df_scrap, ruta_archivo)
    i += 1

  0%|                                                                                                                                           | 0/755 [00:00<?, ?it/s]

https://inmuebles.mercadolibre.com.mx/casas/renta/distrito-federal/cuauhtemoc/
Este es el url en la posicion 0
Scrapeando Página 1
se escrapearon 144 articulos
coorendas de la vivienda 0, de 144 viviendas
coorendas de la vivienda 1, de 144 viviendas
coorendas de la vivienda 2, de 144 viviendas
coorendas de la vivienda 3, de 144 viviendas
coorendas de la vivienda 4, de 144 viviendas
coorendas de la vivienda 5, de 144 viviendas
coorendas de la vivienda 6, de 144 viviendas
coorendas de la vivienda 7, de 144 viviendas
coorendas de la vivienda 8, de 144 viviendas
coorendas de la vivienda 9, de 144 viviendas
coorendas de la vivienda 10, de 144 viviendas
coorendas de la vivienda 11, de 144 viviendas
coorendas de la vivienda 12, de 144 viviendas
coorendas de la vivienda 13, de 144 viviendas
coorendas de la vivienda 14, de 144 viviendas
coorendas de la vivienda 15, de 144 viviendas
coorendas de la vivienda 16, de 144 viviendas
coorendas de la vivienda 17, de 144 viviendas
coorendas de la vivien

  0%|▏                                                                                                                        | 1/755 [5:31:10<4161:46:19, 19870.53s/it]

https://inmuebles.mercadolibre.com.mx/departamentos/venta/distrito-federal/cuauhtemoc/
Este es el url en la posicion 1
Scrapeando Página 1
se escrapearon 192 articulos
coorendas de la vivienda 0, de 192 viviendas
coorendas de la vivienda 1, de 192 viviendas
coorendas de la vivienda 2, de 192 viviendas
coorendas de la vivienda 3, de 192 viviendas
coorendas de la vivienda 4, de 192 viviendas
coorendas de la vivienda 5, de 192 viviendas
coorendas de la vivienda 6, de 192 viviendas
coorendas de la vivienda 7, de 192 viviendas
coorendas de la vivienda 8, de 192 viviendas
coorendas de la vivienda 9, de 192 viviendas
coorendas de la vivienda 10, de 192 viviendas
coorendas de la vivienda 11, de 192 viviendas
coorendas de la vivienda 12, de 192 viviendas
coorendas de la vivienda 13, de 192 viviendas
coorendas de la vivienda 14, de 192 viviendas
coorendas de la vivienda 15, de 192 viviendas
coorendas de la vivienda 16, de 192 viviendas
coorendas de la vivienda 17, de 192 viviendas
coorendas de l

  0%|▎                                                                                                                       | 2/755 [20:10:14<8200:59:31, 39207.93s/it]

https://inmuebles.mercadolibre.com.mx/departamentos/renta/distrito-federal/cuauhtemoc/
Este es el url en la posicion 2
Scrapeando Página 1
se escrapearon 240 articulos
coorendas de la vivienda 0, de 240 viviendas


In [ ]:
import pandas as pd

columnas = [
    'fecha',
    'descripcion',
    'recamaras',
    'banos',
    'superficie_m2',
    'precio_actual',
    'link',
    'imagen',
    'latitud',
    'longitud',
    'tipo_vivienda',
    'operacion',
    'estado',
    'municipio']
    


df = pd.DataFrame(columns=columnas)

print(df)

In [ ]:
df.to_csv('ws_vivienda.csv', index=False)

In [ ]:
url = "https://inmuebles.mercadolibre.com.mx/casas/venta/distrito-federal/alvaro-obregon/"

tipo_vivienda, operacion, estado, municipio = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/([^/]+)/',url).groups()

df['tipo_vivienda'] = tipo_vivienda
df['operacion'] = operacion
df['estado'] = estado
df['municipio'] = municipio

In [ ]:
municipio

In [ ]:
url = 'https://inmuebles.mercadolibre.com.mx/departamentos/venta/sonora/'
tipo_vivienda, operacion, estado, municipio = re.search(r'/(casas|departamentos)/(venta|renta)/([^/]+)/([^/]+)/',url).groups()